In [ ]:
#installing the transformers library
!pip install transformers

In [ ]:
#importing the pretrained biobert tokenizer and the biobert model
from transformers import AutoTokenizer, TFAutoModel
biobert_tokenizer = AutoTokenizer.from_pretrained("cambridgeltl/BioRedditBERT-uncased")

biobert_model = TFAutoModel.from_pretrained("cambridgeltl/BioRedditBERT-uncased")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['embeddings.position_ids']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions without further training.


In [ ]:
#displaying biobert model summary
biobert_model.summary()

Model: "tf_bert_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  108310272 
                                                                 
Total params: 108310272 (413.17 MB)
Trainable params: 108310272 (413.17 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
#loading the json medical data files
import json
with open('intentsindonesian.json') as f1:
  ehealth=json.load(f1)["intents"]

In [ ]:
#function to extract question,answer and tags from the json objects
def extract_responses_patterns_tag(json_data):
  patterns=[]
  responses=[]
  tag=[]
  for i in json_data:
    patterns.append(i['patterns'])
    responses.append(i['responses'])
    tag.append(i['tag'])
  return patterns,responses,tag

In [ ]:
#extracting the question,answer and tags from the medical json objects
all_patterns=[]
all_responses=[]
all_tag=[]
for i in [ehealth]:
  patterns,responses,tag=extract_responses_patterns_tag(i)
  all_patterns.extend(patterns)
  all_responses.extend(responses)
  all_tag.extend(tag)
print(len(all_patterns),len(all_responses),len(all_tag))

100 100 100


In [ ]:
#loading and storing the medical data into dataframe and csv file correspondingly.
import pandas as pd
all_data=pd.DataFrame({'patterns':all_patterns,'responses':all_responses,'tag':all_tag})
all_data.to_csv('all_data.csv',index=False)

In [ ]:
#loading medical data from the disk
all_data=pd.read_csv('all_data.csv')

In [ ]:
#displaying the medical data.
all_data

,patterns,responses,tag
0,"['Hai', 'Hai', 'Apakah ada orang di sana?', 'H...",['Halo. Ceritakan bagaimana perasaanmu hari in...,salam
1,['Selamat pagi'],['Selamat pagi. Saya harap tidur malam Anda ny...,pagi
2,['Selamat siang'],['Selamat siang. Bagaimana harimu?'],siang
3,['Selamat malam'],['Selamat malam. Bagaimana harimu?'],malam
4,['Selamat malam'],"['Selamat malam. Tidur yang nyenyak', 'Selamat...",malam
...,...,...,...
95,['Apa peran terapi seni dalam pengobatan gangg...,['Terapi seni adalah bentuk terapi kreatif yan...,fakta-48
96,['Apa yang dimaksud dengan burnout dan bagaima...,['Burnout adalah kelelahan fisik dan emosional...,fakta-49
97,['Bagaimana penggunaan alkohol dan obat-obatan...,['Penggunaan alkohol dan obat-obatan terlarang...,fakta-50
98,['Hewan peliharaan dapat memberikan dukungan e...,['Hewan peliharaan dapat memberikan dukungan e...,fakta-51


In [ ]:
import re
import numpy as np
#function to preprocess tags post loading from disk
def preprocess_tags(tag):
  tag=tag[1:-1]
  tag=tag.replace("\'",'')
  return np.array(tag.split(', '))

In [ ]:
#preprocessing the tags post loading from disk
all_data['tag']=all_data.tag.apply(lambda x: preprocess_tags(x))

In [ ]:
#displaying the data
all_data.head()

,patterns,responses,tag
0,"['Hai', 'Hai', 'Apakah ada orang di sana?', 'H...",['Halo. Ceritakan bagaimana perasaanmu hari in...,[ala]
1,['Selamat pagi'],['Selamat pagi. Saya harap tidur malam Anda ny...,[ag]
2,['Selamat siang'],['Selamat siang. Bagaimana harimu?'],[ian]
3,['Selamat malam'],['Selamat malam. Bagaimana harimu?'],[ala]
4,['Selamat malam'],"['Selamat malam. Tidur yang nyenyak', 'Selamat...",[ala]


In [ ]:

import re
#preprocessing questions and answers.
def decontractions(phrase):
    """decontracted takes text and convert contractions into natural form.
     ref: https://stackoverflow.com/questions/19790188/expanding-english-language-contractions-in-python/47091490#47091490"""
    phrase = re.sub(r"won\'t", "will not", phrase)
    phrase = re.sub(r"can\'t", "can not", phrase)
    phrase = re.sub(r"won\’t", "will not", phrase)
    phrase = re.sub(r"can\’t", "can not", phrase)
    phrase = re.sub(r"n\'t", " not", phrase)
    phrase = re.sub(r"\'re", " are", phrase)
    phrase = re.sub(r"\'s", " is", phrase)
    phrase = re.sub(r"\'d", " would", phrase)
    phrase = re.sub(r"\'ll", " will", phrase)
    phrase = re.sub(r"\'t", " not", phrase)
    phrase = re.sub(r"\'ve", " have", phrase)
    phrase = re.sub(r"\'m", " am", phrase)

    phrase = re.sub(r"n\’t", " not", phrase)
    phrase = re.sub(r"\’re", " are", phrase)
    phrase = re.sub(r"\’s", " is", phrase)
    phrase = re.sub(r"\’d", " would", phrase)
    phrase = re.sub(r"\’ll", " will", phrase)
    phrase = re.sub(r"\’t", " not", phrase)
    phrase = re.sub(r"\’ve", " have", phrase)
    phrase = re.sub(r"\’m", " am", phrase)

    return phrase


def preprocess(text):
    # convert all the text into lower letters
    # remove the words betweent brakets ()
    # remove these characters: {'$', ')', '?', '"', '’', '.',  '°', '!', ';', '/', "'", '€', '%', ':', ',', '('}
    # replace these spl characters with space: '\u200b', '\xa0', '-', '/'

    text = text.lower()
    text = decontractions(text)
    text = re.sub('[$)\?"’.°!;\'€%:,(/]', '', text)
    text = re.sub('\u200b', ' ', text)
    text = re.sub('\xa0', ' ', text)
    text = re.sub('-', ' ', text)
    return text


all_data['preprocessed_question'] = all_data['patterns'].apply(preprocess)
all_data['preprocessed_answer'] = all_data['responses'].apply(preprocess)
all_data.head()


,patterns,responses,tag,preprocessed_question,preprocessed_answer
0,"['Hai', 'Hai', 'Apakah ada orang di sana?', 'H...",['Halo. Ceritakan bagaimana perasaanmu hari in...,[ala],[hai hai apakah ada orang di sana halo halo ha...,[halo ceritakan bagaimana perasaanmu hari ini ...
1,['Selamat pagi'],['Selamat pagi. Saya harap tidur malam Anda ny...,[ag],[ iselamat pagi],[ iselamat pagi saya harap tidur malam anda ny...
2,['Selamat siang'],['Selamat siang. Bagaimana harimu?'],[ian],[ iselamat siang],[ iselamat siang bagaimana harimu]
3,['Selamat malam'],['Selamat malam. Bagaimana harimu?'],[ala],[ iselamat malam],[ iselamat malam bagaimana harimu]
4,['Selamat malam'],"['Selamat malam. Tidur yang nyenyak', 'Selamat...",[ala],[ iselamat malam],[ iselamat malam tidur yang nyenyak iselamat ...


In [ ]:
#finding the lens of preprocessed questions and answers
all_data['question_len']=all_data['preprocessed_question'].apply(lambda x: len(x.split(' ')))
all_data['answer_len']=all_data['preprocessed_answer'].apply(lambda x: len(x.split(' ')))

In [ ]:
#printing different percentiles of question and answer lengths
import numpy as np
for i in range(0,101,10):
  print(i,np.percentile(all_data.question_len,i),np.percentile(all_data.answer_len,i))

0 1.0 4.0
10 5.0 13.9
20 6.0 19.0
30 7.699999999999999 23.7
40 9.0 26.6
50 10.0 29.0
60 11.0 34.8
70 14.0 40.599999999999994
80 18.0 48.2
90 25.0 102.30000000000011
100 38.0 329.0


In [ ]:
#printing different percentiles of question and answer lengths
for i in range(90,101,1):
  print(i,np.percentile(all_data.question_len,i),np.percentile(all_data.answer_len,i))

90 25.0 102.30000000000011
91 25.0 116.34000000000009
92 25.08 140.95999999999998
93 26.140000000000015 152.7700000000001
94 28.119999999999976 163.29999999999995
95 30.049999999999997 169.19999999999993
96 31.0 192.0
97 31.060000000000002 192.84000000000003
98 33.0 220.79999999999984
99 33.050000000000026 260.69000000000034
100 38.0 329.0


In [ ]:
#since more than 100% percent of questions and about 99% of all answers lie within 500 words, we use 500 as the max length
all_data['short_question']=all_data.apply(lambda x: ' '.join(x.preprocessed_question.split(' ')[:500]) if x.question_len>500 else x.preprocessed_question ,axis=1)
all_data['short_answer']=all_data.apply(lambda x: ' '.join(x.preprocessed_answer.split(' ')[:500]) if x.answer_len>500 else x.preprocessed_answer ,axis=1)

In [ ]:
#displaying the data post truncating the question and answers to length of 500
all_data.head(5)

,patterns,responses,tag,preprocessed_question,preprocessed_answer,question_len,answer_len,short_question,short_answer
0,"['Hai', 'Hai', 'Apakah ada orang di sana?', 'H...",['Halo. Ceritakan bagaimana perasaanmu hari in...,[ala],[hai hai apakah ada orang di sana halo halo ha...,[halo ceritakan bagaimana perasaanmu hari ini ...,18,42,[hai hai apakah ada orang di sana halo halo ha...,[halo ceritakan bagaimana perasaanmu hari ini ...
1,['Selamat pagi'],['Selamat pagi. Saya harap tidur malam Anda ny...,[ag],[ iselamat pagi],[ iselamat pagi saya harap tidur malam anda ny...,3,14,[ iselamat pagi],[ iselamat pagi saya harap tidur malam anda ny...
2,['Selamat siang'],['Selamat siang. Bagaimana harimu?'],[ian],[ iselamat siang],[ iselamat siang bagaimana harimu],3,5,[ iselamat siang],[ iselamat siang bagaimana harimu]
3,['Selamat malam'],['Selamat malam. Bagaimana harimu?'],[ala],[ iselamat malam],[ iselamat malam bagaimana harimu],3,5,[ iselamat malam],[ iselamat malam bagaimana harimu]
4,['Selamat malam'],"['Selamat malam. Tidur yang nyenyak', 'Selamat...",[ala],[ iselamat malam],[ iselamat malam tidur yang nyenyak iselamat ...,3,11,[ iselamat malam],[ iselamat malam tidur yang nyenyak iselamat ...


In [ ]:
#preparing the positive label dataset by taking the original question and original answer
all_dataset=all_data[['short_question','short_answer','tag']].copy()
all_dataset['label']=1.0


In [ ]:
#displaying the positive labelled dataset
all_dataset.head(3)


,short_question,short_answer,tag,label
0,[hai hai apakah ada orang di sana halo halo ha...,[halo ceritakan bagaimana perasaanmu hari ini ...,[ala],1.0
1,[ iselamat pagi],[ iselamat pagi saya harap tidur malam anda ny...,[ag],1.0
2,[ iselamat siang],[ iselamat siang bagaimana harimu],[ian],1.0


In [ ]:
#function to extract the negative samples using tags
def extract_negative_samples(patterns,tag):
  stop=False
  while (not stop):
    sample_row= all_dataset.sample()
    sample_tag=sample_row.tag.values[0]
    inter_tag=set(tag[0]).intersection(set(sample_tag))
    if len(inter_tag)==0:
      stop=True
  return sample_row

In [ ]:
#preparing the negative labelled dataset
from tqdm.notebook import tqdm
tqdm.pandas()
negative_labels=all_dataset.progress_apply(lambda x: pd.Series([x.short_question,extract_negative_samples(x.short_question,x.tag).short_answer.values[0],x.tag]),axis=1)
negative_labels['label']=-1.0
negative_labels.columns=['short_question','short_answer','tag','label']

  0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
#displaying the negative labelled dataset
negative_labels

,short_question,short_answer,tag,label
0,[hai hai apakah ada orang di sana halo halo ha...,[banyak orang menganggap dukungan sejawat seba...,[ala],-1.0
1,[ iselamat pagi],[gangguan bipolar adalah gangguan mental yang ...,[ag],-1.0
2,[ iselamat siang],[ isaya senang anda menganggap ini berguna ada...,[ian],-1.0
3,[ iselamat malam],[ iselamat pagi saya harap tidur malam anda ny...,[ala],-1.0
4,[ iselamat malam],[banyak orang menganggap dukungan sejawat seba...,[ala],-1.0
...,...,...,...,...
95,[apa peran terapi seni dalam pengobatan ganggu...,[banyak orang menganggap dukungan sejawat seba...,[akta-4],-1.0
96,[apa yang dimaksud dengan burnout dan bagaiman...,[ isaya senang anda menganggap ini berguna ada...,[akta-4],-1.0
97,[bagaimana penggunaan alkohol dan obat obatan ...,[ amerasa nyaman dengan profesional yang beker...,[akta-5],-1.0
98,[hewan peliharaan dapat memberikan dukungan em...,[ isumber informasi terbaik mengenai obat obat...,[akta-5],-1.0


In [ ]:
#concatenating the positive and negative labelled dataset to get the final labelled dataset.
all_data_with_labels=pd.concat([all_dataset,negative_labels],axis=0)
all_data_with_labels.shape

(200, 4)

In [ ]:
#splitting the data into train and validation
from sklearn.model_selection import train_test_split
train, validation = train_test_split(all_data_with_labels, test_size=0.2,random_state=42,shuffle=True,stratify=all_data_with_labels.label)

In [ ]:
#saving train and validation data to disk
train.to_csv('train_data_chatbot.csv',index=False)
validation.to_csv('validation_data_chatbot.csv',index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
cd 'drive/My Drive/chatbot'

In [ ]:
#loading train and validation data from disk
import pandas as pd
train=pd.read_csv('train_data_chatbot.csv')
validation=pd.read_csv('validation_data_chatbot.csv')

In [ ]:
#displaying the train label counts
train.label.value_counts()

label
-1.0    80
 1.0    80
Name: count, dtype: int64

In [ ]:
#displaying the validation label counts
validation.label.value_counts()

label
-1.0    20
 1.0    20
Name: count, dtype: int64

In [ ]:
#displaying the train dataset
train.head(5)

,short_question,short_answer,tag,label
0,[apa itu terapi apakah saya memerlukan terapi ...,[ amendukung remaja yang mengalami masalah kes...,['akta-'],-1.0
1,[apa yang harus saya lakukan jika saya mengkha...,[yang paling penting adalah berbicara dengan s...,['akta-2'],1.0
2,[apa perbedaan antara kecemasan dan stres],[ notidur yang cukup sangat penting untuk kese...,['akta-3'],-1.0
3,[hai hai apakah ada orang di sana halo halo ha...,[halo ceritakan bagaimana perasaanmu hari ini ...,['ala'],1.0
4,[apa sajakah tanda tanda peringatan penyakit m...,[gejala gangguan kesehatan mental bervariasi t...,['akta-1'],1.0


In [ ]:
#extracting the short answers and questions from the train data
questions=train['short_question']
answers=train['short_answer']
labels=train['label']

In [ ]:
#https://blog.tensorflow.org/2019/05/transformer-chatbot-tutorial-with-tensorflow-2.html
#Setting Max_length to be 512 as discussed above
MAX_LENGTH = 512
import tensorflow as tf


# Tokenize, filter and pad sentences
def tokenize_and_filter(questions, answers):
  tokenized_questions, tokenized_answers = [], []

  for (question, answer) in zip(questions, answers):
    # generating sequences
    tokenized_question =  biobert_tokenizer.encode(question)
    tokenized_answer = biobert_tokenizer.encode(answer)
    tokenized_questions.append(tokenized_question)
    tokenized_answers.append(tokenized_answer)

  # padding the sequences
  tokenized_questions = tf.keras.preprocessing.sequence.pad_sequences(
      tokenized_questions, maxlen=MAX_LENGTH, padding='post')
  tokenized_answers = tf.keras.preprocessing.sequence.pad_sequences(
      tokenized_answers, maxlen=MAX_LENGTH, padding='post')

  return tokenized_questions, tokenized_answers

#tokenizing and padding the train questions and answers
questions, answers = tokenize_and_filter(questions, answers)

In [ ]:
#preparing the question mask and the answer mask of the train dataset
train_question_mask=[[1 if token!=0 else 0 for token in question] for question in questions]
train_answer_mask=[[1 if token!=0 else 0 for token in answer] for answer in answers]


In [ ]:
#extracting the short answers and questions from the validation data
val_questions=validation['short_question']
val_answers=validation['short_answer']
val_labels=validation['label']

In [ ]:
#tokenizing and padding the validation question and answers
val_questions, val_answers = tokenize_and_filter(val_questions, val_answers)

In [ ]:
#preparing the question and answer mask for the validation data
val_question_mask=[[1 if token!=0 else 0 for token in question] for question in val_questions]
val_answer_mask=[[1 if token!=0 else 0 for token in answer] for answer in val_answers]


In [ ]:
#https://blog.tensorflow.org/2019/05/transformer-chatbot-tutorial-with-tensorflow-2.html
#preparing the training  dataset
import tensorflow as tf
BATCH_SIZE = 2
BUFFER_SIZE = 20000
dataset = tf.data.Dataset.from_tensor_slices((
    {
        'question': questions,
        'answer': answers,
        'question_mask':train_question_mask,
        'answer_mask':train_answer_mask
    },
    {
        'label': labels.values
    },
))


In [ ]:
# dataset = dataset.cache()
dataset = dataset.shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE,drop_remainder=True)
dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)

In [ ]:
#https://blog.tensorflow.org/2019/05/transformer-chatbot-tutorial-with-tensorflow-2.html
#preparing the validation dataset
import tensorflow as tf
BATCH_SIZE = 2
BUFFER_SIZE = 20000
val_dataset = tf.data.Dataset.from_tensor_slices((
    {
        'question': val_questions,
        'answer': val_answers,
        'question_mask':val_question_mask,
        'answer_mask':val_answer_mask
    },
    {
        'label': val_labels.values
    },
))

val_dataset = val_dataset.cache()
val_dataset = val_dataset.shuffle(BUFFER_SIZE)
val_dataset = val_dataset.batch(BATCH_SIZE,drop_remainder=True)
val_dataset = val_dataset.prefetch(tf.data.experimental.AUTOTUNE)

In [ ]:
#creating the ffn layer
#https://github.com/ash3n/DocProduct/blob/master/docproduct/models.py
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input,Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Concatenate,Conv1D,MaxPool1D,Dropout
class FFN(tf.keras.layers.Layer):
    def __init__(
            self,
            name='FFN',
            **kwargs):
        """Simple Dense wrapped with various layers
        """

        super(FFN, self).__init__(name=name, **kwargs)
        self.dropout = 0.2
        self.ffn_layer = tf.keras.layers.Dense(
            units=768,
            activation='relu',
            kernel_initializer=tf.keras.initializers.glorot_normal(seed=32),name='FC1')


    def call(self, inputs):
        ffn_embedding = self.ffn_layer(inputs)
        ffn_embedding = tf.keras.layers.Dropout(
            self.dropout)(ffn_embedding)
        ffn_embedding += inputs
        return ffn_embedding

In [ ]:
#displaying the biobert model summary
biobert_model.summary()

Model: "tf_bert_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  108310272 
                                                                 
Total params: 108310272 (413.17 MB)
Trainable params: 108310272 (413.17 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
#creating the medicalbert model
#https://github.com/ash3n/DocProduct/blob/master/docproduct/models.py
class IDMentalBERT(tf.keras.Model):
    def __init__(
            self,
            trainable=False,
            name=''):
        super(IDMentalBERT, self).__init__(name=name)

        self.q_ffn_layer = FFN(name='q_ffn')
        self.a_ffn_layer = FFN(name='a_ffn')
        self.biobert_model=biobert_model
        self.biobert_model.trainable=trainable
        self.cos=tf.keras.layers.Dot(axes=1,normalize=True)

    def call(self, inputs):
      question_embeddings=self.biobert_model(input_ids=inputs['question'],attention_mask=inputs['question_mask']).pooler_output
      answer_embeddings=self.biobert_model(input_ids=inputs['answer'],attention_mask=inputs['answer_mask']).pooler_output
      q_ffnn=self.q_ffn_layer(question_embeddings)
      a_ffnn=self.a_ffn_layer(answer_embeddings)
      output=self.cos([q_ffnn,a_ffnn])
      return {"label":output}


In [ ]:
#defining custom callback
class custom_callback(tf.keras.callbacks.Callback):
  def on_train_begin(self, logs={}):
        ## on begin of training, we are creating a instance varible called history
        ## it is a dict with keys [loss, acc, val_loss, val_acc]
        self.history={'acc': []}

  def on_epoch_end(self, epoch, logs={}):
        self.history['acc'].append(logs.get('custom_metric_acc'))



        #saving the model if validation accuracy increased from previous epoch
        if  (epoch==0) or (logs.get('custom_metric_acc')>self.history['acc'][epoch-1]):
          self.model.save_weights('medical_bert_re'+str(epoch)+'/medic'+str(epoch)+'_'+str(logs.get('custom_metric_acc')))


        is_nan_values=[]
        for i in self.model.get_weights():
          is_nan_values.append(np.isnan(i).any())

        #stopping the training if weights is nan or loss is nan or inf
        if (np.array(is_nan_values).any() or (tf.math.is_nan(logs.get('loss'))) or (np.isinf(logs.get('loss')))):
          self.model.stop_training = True



In [ ]:
#defining the custom metric
batch_size=2
def custom_metric_acc(y_true,y_pred):
  y_true=tf.reshape(y_true,[tf.constant(batch_size)])
  y_pred=tf.reshape(y_pred,[tf.constant(batch_size)])
  c=tf.constant(0,dtype='float32')
  d=tf.cast(tf.math.greater_equal(y_true,c),dtype='float32')
  e=tf.cast(tf.math.greater_equal(y_pred,c),dtype='float32')
  f=tf.cast(tf.math.equal(d,e),dtype='float32')
  g=tf.reduce_sum(f)
  h=tf.cast(tf.shape(f),dtype='float32')
  i=g/h
  return i

In [ ]:
#compiling and training the medical qa model
import tensorflow.keras.backend as K
K.set_floatx('float32')
learning_rate=5e-6
num_epochs=5
optimizer = tf.keras.optimizers.Adam(lr=learning_rate)
medical_qa_model=IDMentalBERT(trainable=True)
medical_qa_model.compile(
    optimizer=optimizer, loss=tf.keras.losses.mean_squared_error, metrics=[custom_metric_acc])

epochs = num_epochs

medical_qa_model.fit(dataset,validation_data=val_dataset,epochs=epochs,callbacks=[custom_callback()])
medical_qa_model.summary()

Epoch 1/5
80/80 [==============================] - 2171s 26s/step - loss: 1.0406 - custom_metric_acc: 0.4812 - val_loss: 1.0004 - val_custom_metric_acc: 0.5000
Epoch 2/5
80/80 [==============================] - 2127s 27s/step - loss: 1.0012 - custom_metric_acc: 0.4875 - val_loss: 1.0022 - val_custom_metric_acc: 0.5000
Epoch 3/5
80/80 [==============================] - 2051s 26s/step - loss: 1.0184 - custom_metric_acc: 0.4313 - val_loss: 1.0004 - val_custom_metric_acc: 0.5000
Epoch 4/5
80/80 [==============================] - 2064s 26s/step - loss: 1.0010 - custom_metric_acc: 0.5250 - val_loss: 1.0001 - val_custom_metric_acc: 0.5000
Epoch 5/5
80/80 [==============================] - 2030s 25s/step - loss: 0.9997 - custom_metric_acc: 0.5250 - val_loss: 1.0003 - val_custom_metric_acc: 0.5000
Model: ""
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 q_ffn (FFN)                 multiple                  590

In [ ]:
import tensorflow.keras.backend as K
K.set_floatx('float32')
medical_qa_model=IDMentalBERT(trainable=True)
medical_qa_model.load_weights('./medical_bert_re3/medic3_0.4937500059604645')
learning_rate=5e-6
optimizer = tf.keras.optimizers.Adam(lr=learning_rate)
medical_qa_model.compile(
    optimizer=optimizer, loss=tf.keras.losses.mean_squared_error, metrics=[custom_metric_acc])

# original_labels=[]
predicted_labels=[]
from tqdm.notebook import tqdm
for i in tqdm(range(len(val_questions))):
  predicted_labels.append(medical_qa_model.predict({'question':np.array([val_questions[i]]),'question_mask':np.array([val_question_mask[i]]),
                             'answer':np.array([val_answers[i]]),'answer_mask':np.array([val_answer_mask[i]])})['label'][0][0])

NotFoundError: Unsuccessful TensorSliceReader constructor: Failed to find any matching files for ./medical_bert_re3/medic3_0.4937500059604645

In [ ]:
#defining function to return accuracy, correctly predicted positive points and correctly predicted negative points
def acc_threshold(predicted_labels,threshold):
  correct_pred_pos=[]
  correct_pred_neg=[]
  for i in range(len(predicted_labels)):
    if predicted_labels[i]==threshold and val_labels[i]==1:
      correct_pred_pos.append(predicted_labels[i])
  acc=len(correct_pred_pos+correct_pred_neg)/len(predicted_labels)
  print('accuracy for threshold',threshold,'is',acc)
  return (acc,correct_pred_pos,correct_pred_neg)

In [ ]:
#function to print the best accuracy and the threshold at which it is occuring
def best_threshold_acc(predicted_labels):
  acc=[]
  thresholds=np.array(range(-10,11,1))*0.1
  for i in tqdm(range(-10,11,1)):
    acc.append(acc_threshold(predicted_labels,i*0.1)[0])
  print('best accuracy is',max(acc),'at threshold',thresholds[np.argmax(acc)])

In [ ]:
#find best threshold and accuracy of the model with best train accuracy on validation data
best_threshold_acc(predicted_labels)

NameError: name 'predicted_labels' is not defined

In [ ]:
#displaying the pdf of correctly predicted positive and negative points
import seaborn as sns
_,correct_pred_pos,correct_pred_neg=acc_threshold(predicted_labels,0.2)
sns.distplot(correct_pred_pos)
sns.distplot(correct_pred_neg)

NameError: name 'predicted_labels' is not defined

In [ ]:
#evaluating the model on validation data
import warnings
warnings.filterwarnings('ignore')
import tensorflow.keras.backend as K
import tensorflow as tf
import numpy as np
K.set_floatx('float32')
medical_qa_model=IDMentalBERT(trainable=False)
medical_qa_model.load_weights('/medical_bert_re4/medic4_0.512499988079071')
learning_rate=5e-6
optimizer = tf.keras.optimizers.Adam(lr=learning_rate)
predicted_labels=[]
from tqdm.notebook import tqdm
for i in tqdm(range(len(val_questions))):
  predicted_labels.append(medical_qa_model.predict({'question':np.array([val_questions[i]]),'question_mask':np.array([val_question_mask[i]]),
                             'answer':np.array([val_answers[i]]),'answer_mask':np.array([val_answer_mask[i]])})['label'][0][0])


NotFoundError: Unsuccessful TensorSliceReader constructor: Failed to find any matching files for /medical_bert_re4/medic4_0.512499988079071